In [346]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
con = duckdb.connect("warehouse.duckdb")

In [347]:
con.execute("""
    DROP TABLE IF EXISTS fact_sales;
    DROP TABLE IF EXISTS dim_time;
    DROP TABLE IF EXISTS dim_item;
    DROP TABLE IF EXISTS dim_location;
            
    -- DIMENSIONES — atributos descriptivos y jerarquías
    CREATE TABLE dim_time (
    time_key INTEGER PRIMARY KEY,
    day DATE, month VARCHAR, quarter VARCHAR, year INTEGER );
            
    CREATE TABLE dim_item (
    item_key INTEGER PRIMARY KEY,
    item_name VARCHAR, brand VARCHAR, category VARCHAR );
            
    CREATE TABLE dim_location (
    loc_key INTEGER PRIMARY KEY,
    city VARCHAR, state VARCHAR, country VARCHAR );
    -- HECHOS — claves foraneas + medidas numericas
            
    CREATE TABLE fact_sales (
    time_key INTEGER REFERENCES dim_time(time_key),
    item_key INTEGER REFERENCES dim_item(item_key),
    loc_key INTEGER REFERENCES dim_location(loc_key),
    dollars_sold DECIMAL(12,2),
    units_sold INTEGER );
""")

In [348]:
con.execute("""
    INSERT INTO dim_time SELECT * FROM read_csv_auto('data/dim_time.csv');
    INSERT INTO dim_item SELECT * FROM read_csv_auto('data/dim_item.csv');
    INSERT INTO dim_location SELECT * FROM read_csv_auto('data/dim_location.csv');
    INSERT INTO fact_sales SELECT * FROM read_csv_auto('data/fact_sales.csv');
""")

In [349]:
con.execute("""
    DROP VIEW IF EXISTS v_sales;
    CREATE VIEW v_sales AS
    SELECT t.year, t.quarter, t.month, i.category, i.item_name, l.country, f.dollars_sold, f.units_sold
    FROM fact_sales f
    JOIN dim_time t USING (time_key)
    JOIN dim_item i USING (item_key)
    JOIN dim_location l USING (loc_key);
""")

Roll-up (drill-up) y Drill-down

In [350]:
con.execute("""
    SELECT country, SUM(dollars_sold), SUM(units_sold) FROM v_sales GROUP BY country;
    SELECT year, month, quarter, SUM(dollars_sold), SUM(units_sold) FROM v_sales
    GROUP BY year, month, quarter;
""")

Slice & dice

In [351]:
con.execute("""
    SELECT category, SUM(dollars_sold), SUM(units_sold) FROM v_sales
    WHERE country = 'USA' GROUP BY category;
""")

Pivot (rotate)

In [352]:
con.execute("""
    PIVOT v_sales ON country USING SUM(dollars_sold) GROUP BY category;
""")

In [353]:
cube_df = con.execute("""
    SELECT t.year, l.country, i.category,
        SUM(f.dollars_sold) AS dollars_sold,
        SUM(f.units_sold) AS units_sold
    FROM fact_sales f
    JOIN dim_time t ON f.time_key = t.time_key
    JOIN dim_location l ON f.loc_key = l.loc_key
    JOIN dim_item i ON f.item_key = i.item_key
    GROUP BY CUBE (t.year, l.country, i.category)
""").df()
cube_df


,year,country,category,dollars_sold,units_sold
0,<NA>,NaN,NaN,1087931.42,38896.0
1,2025,NaN,NaN,1087931.42,38896.0
2,2025,USA,NaN,371309.46,11613.0
3,2025,Mexico,NaN,382387.89,14230.0
4,2025,Chile,NaN,149523.18,5246.0
5,2025,Colombia,NaN,184710.89,7807.0
6,2025,Mexico,Electronics,51265.67,1865.0
7,2025,Chile,Grocery,46117.75,1632.0
8,2025,USA,Beauty,78809.76,2665.0
9,2025,Chile,Home,25176.05,822.0


In [354]:
rollup_df = con.execute("""
    SELECT year, country, category, SUM(dollars_sold) AS dollars_sold, SUM(units_sold) AS units_sold
    FROM v_sales
    GROUP BY ROLLUP (year, country, category)
""").df()
rollup_df

,year,country,category,dollars_sold,units_sold
0,<NA>,NaN,NaN,1087931.42,38896.0
1,2025,NaN,NaN,1087931.42,38896.0
2,2025,Chile,NaN,149523.18,5246.0
3,2025,Mexico,NaN,382387.89,14230.0
4,2025,USA,NaN,371309.46,11613.0
5,2025,USA,Sports,65871.35,2004.0
6,2025,Mexico,Home,61481.04,2217.0
7,2025,Chile,Electronics,19559.61,686.0
8,2025,Mexico,Grocery,122737.79,4596.0
9,2025,Colombia,NaN,184710.89,7807.0


¿Cuál es más rápida? Mide con %timeit o time.perf_counter().
• ¿Cuál se puede particionar? ¿Por qué SUM se puede agregar incrementalmente y MEDIAN no?
• Si tuvieras 1,000 millones de filas, ¿cuál materializarías y cuál calcularías al vuelo? ¿Por qué?

In [355]:
groupset_sums_df = con.execute("""
    SELECT year, country, SUM(dollars_sold), SUM(units_sold)
    FROM v_sales
    GROUP BY GROUPING SETS ((year), (country), (year,country), ());
""").df()
groupset_sums_df

,year,country,sum(dollars_sold),sum(units_sold)
0,2025,NaN,1087931.42,38896.0
1,<NA>,USA,371309.46,11613.0
2,<NA>,Colombia,184710.89,7807.0
3,2025,Mexico,382387.89,14230.0
4,2025,USA,371309.46,11613.0
5,<NA>,Mexico,382387.89,14230.0
6,2025,Chile,149523.18,5246.0
7,<NA>,NaN,1087931.42,38896.0
8,2025,Colombia,184710.89,7807.0
9,<NA>,Chile,149523.18,5246.0


In [356]:
groupset_median_df = con.execute("""
    SELECT year, country,   MEDIAN(dollars_sold), MEDIAN(units_sold)
    FROM v_sales
    GROUP BY GROUPING SETS ((year), (country), (year,country), ());
""").df()
groupset_median_df

,year,country,median(dollars_sold),median(units_sold)
0,<NA>,Colombia,75.83,3.0
1,<NA>,Chile,93.42,3.0
2,<NA>,USA,104.48,3.0
3,2025,Chile,93.42,3.0
4,2025,USA,104.48,3.0
5,2025,Colombia,75.83,3.0
6,2025,Mexico,90.17,4.0
7,2025,NaN,91.31,3.0
8,<NA>,Mexico,90.17,4.0
9,<NA>,NaN,91.31,3.0


In [ ]:
iceberg_df = con.execute("""
        select country, month, item_name, SUM(dollars_sold) AS dollars_sold
        FROM v_sales 
        Group by cube (country, month, item_name)
        having SUM(dollars_sold) >= 50000
""").df()
iceberg_df


,country,month,item_name,dollars_sold
0,NaN,4,None,91053.99
1,NaN,3,None,91386.51
2,USA,NaN,None,371309.46
3,NaN,6,None,89586.91
4,NaN,7,None,91463.75
5,NaN,5,None,85397.28
6,NaN,9,None,84327.79
7,NaN,2,None,81791.69
8,Chile,NaN,None,149523.18
9,NaN,1,None,83436.45
